In [1]:
#!pip install -qqq arize-otel

In [2]:
import os
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]


In [3]:
project_name = "span_findata_briefing"

In [4]:
# Import open-telemetry dependencies
from arize.otel import register
import pandas as pd
import json

# Setup OTel via our convenience function
tracer_provider = register(
    space_id = space_id, # in app space settings page
    api_key = api_key, # in app space settings page
    project_name = project_name,
)


🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: span_findata_briefing
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# Briefing Data Configuration
# Verticals, wires (headline, body, time), aggregated briefings (header + bullets), account personas, personalized (bullets)

import random

BRIEFING_DATA_CONFIG = {
    "Technology": {
        "wires": [
            ("Big Tech earnings beat estimates", "Apple, Microsoft, and Google reported stronger-than-expected Q3 results. Cloud and AI demand drove growth.", "08:42"),
            ("Chip shortage eases", "Semiconductor supply constraints have improved; automakers and device makers see relief by year-end.", "09:15"),
            ("AI regulation draft", "EU and US lawmakers are drafting rules for generative AI; industry seeks clarity on compliance.", "10:03"),
        ],
        "aggregated_briefing": "--- Technology | Morning Brief ---\n• Earnings: Major tech beat; cloud and AI driving growth.\n• Supply: Semiconductor constraints easing; relief by year-end.\n• Regulation: EU/US drafting AI rules; compliance clarity sought.\nBottom line: Tech fundamentals supportive; watch regulatory headlines.",
        "accounts": ["Tech Growth Fund", "Software Equity Partners", "Infrastructure Holdings"],
        "account_context": {
            "Tech Growth Fund": "Growth equity, large-cap tech, cloud and AI focus.",
            "Software Equity Partners": "SaaS and enterprise software, compliance and regulation.",
            "Infrastructure Holdings": "Data centers, semiconductors, infrastructure spend.",
        },
        "personalized": {
            "Tech Growth Fund": "**Tech Growth Fund**\n• Overweight large-cap tech; earnings support valuations.\n• Favor cloud and AI exposure.",
            "Software Equity Partners": "**Software Equity Partners**\n• SaaS/enterprise benefited from spend; monitor AI regulation for compliance.\n• Same factual briefing; emphasize software and compliance angles.",
            "Infrastructure Holdings": "**Infrastructure Holdings**\n• Data center and semi supply improvement positive for infra names.\n• Watch permitting and capex for build-out.",
        },
    },
    "Healthcare": {
        "wires": [
            ("FDA approves new drug", "The FDA granted approval for a novel therapy; analysts see blockbuster potential.", "08:38"),
            ("Hospital staffing costs rise", "Labor costs continued to pressure hospital margins in Q3.", "09:22"),
            ("Biotech M&A activity", "Several mid-cap biotechs are in play; deal flow expected to pick up.", "10:11"),
        ],
        "aggregated_briefing": "--- Healthcare | Morning Brief ---\n• Approval: FDA granted novel therapy; blockbuster potential.\n• Margins: Hospital labor costs pressuring Q3 margins.\n• M&A: Mid-cap biotech deal flow picking up.\nBottom line: Stock-specific; approvals and M&A positive, labor a headwind for services.",
        "accounts": ["Healthcare Value Fund", "Biotech Growth", "MedTech Investors"],
        "account_context": {
            "Healthcare Value Fund": "Value-oriented, hospitals and services, margin focus.",
            "Biotech Growth": "Late-stage and commercial biotech, M&A and approvals.",
            "MedTech Investors": "Devices and diagnostics, hospital capital spending.",
        },
        "personalized": {
            "Healthcare Value Fund": "**Healthcare Value Fund**\n• Value names in hospitals/services may see margin headwinds.\n• Selectivity on labor-exposed names.",
            "Biotech Growth": "**Biotech Growth**\n• New approval and M&A support biotech; focus late-stage and commercial.\n• Deal flow key theme.",
            "MedTech Investors": "**MedTech Investors**\n• MedTech demand stable; watch hospital capex given labor pressure.\n• Devices and diagnostics in focus.",
        },
    },
    "Energy": {
        "wires": [
            ("Oil prices volatile", "Crude swung on demand concerns and OPEC+ supply signals; inventories tightened.", "08:45"),
            ("Renewables capex", "Utilities and oil majors increased renewables investment; permitting delays persist.", "09:30"),
            ("Natural gas storage", "Storage levels above five-year average; winter demand outlook mixed.", "10:00"),
        ],
        "aggregated_briefing": "--- Energy | Morning Brief ---\n• Oil: Volatility on demand and OPEC+; inventories tightened.\n• Renewables: Capex up; permitting delays persist.\n• Gas: Storage above 5yr avg; winter demand mixed.\nBottom line: Oil tradeable; renewables theme intact but permitting risk.",
        "accounts": ["Energy Income Fund", "Clean Energy Partners", "Commodity Fund"],
        "account_context": {
            "Energy Income Fund": "Dividend and income, oil and gas, free cash flow.",
            "Clean Energy Partners": "Renewables, permitting, grid and storage.",
            "Commodity Fund": "Oil, gas, commodities, trading and storage.",
        },
        "personalized": {
            "Energy Income Fund": "**Energy Income Fund**\n• Dividend names in oil/gas supported by volatility; focus FCF.\n• Storage and winter demand for gas.",
            "Clean Energy Partners": "**Clean Energy Partners**\n• Renewables capex positive; permitting and grid build-out key.\n• Same briefing; emphasize clean energy angles.",
            "Commodity Fund": "**Commodity Fund**\n• Oil and gas volatility creates trading opportunities.\n• Storage data key for gas view.",
        },
    },
    "Financials": {
        "wires": [
            ("Fed holds rates", "The Fed left rates unchanged; dot plot suggests one cut by year-end.", "08:50"),
            ("Bank earnings", "Major banks reported mixed results; NII pressure, trading revenue stronger.", "09:18"),
            ("Insurance claims", "Catastrophe claims weighed on P&C insurers; reinsurance renewals in focus.", "10:05"),
        ],
        "aggregated_briefing": "--- Financials | Morning Brief ---\n• Fed: On hold; dot plot implies one cut by year-end.\n• Banks: Mixed results; NII pressure, trading stronger.\n• P&C: Cat claims weighed; reinsurance renewals in focus.\nBottom line: Rates path and NII in focus for banks; P&C watch renewals.",
        "accounts": ["Bank Equity Fund", "Insurance Holdings", "Macro Fund"],
        "account_context": {
            "Bank Equity Fund": "Universal banks, NII, credit quality, trading.",
            "Insurance Holdings": "P&C and life, cat claims, reinsurance renewals.",
            "Macro Fund": "Rates, Fed, labor and inflation data.",
        },
        "personalized": {
            "Bank Equity Fund": "**Bank Equity Fund**\n• NII and credit quality in focus; trading supportive for universal banks.\n• Dot plot and funding costs matter.",
            "Insurance Holdings": "**Insurance Holdings**\n• Cat claims and reinsurance renewals drive P&C; life stable.\n• Renewals key for pricing.",
            "Macro Fund": "**Macro Fund**\n• Rates path and Fed communication key; watch dot plot and labor.\n• One cut by year-end baseline.",
        },
    },
    "Consumer": {
        "wires": [
            ("Retail sales beat", "September retail sales came in above consensus; back-to-school and discretionary strength.", "08:35"),
            ("Travel demand strong", "Airlines and hotels reported robust demand; international travel recovering.", "09:40"),
            ("Housing affordability", "Mortgage rates and prices weigh on affordability; existing home sales soft.", "10:12"),
        ],
        "aggregated_briefing": "--- Consumer | Morning Brief ---\n• Retail: September beat; back-to-school and discretionary strength.\n• Travel: Airlines and hotels robust; international recovery.\n• Housing: Affordability headwind; existing home sales soft.\nBottom line: Discretionary and travel supportive; housing under pressure.",
        "accounts": ["Consumer Discretionary Fund", "Travel & Leisure", "Housing Focus"],
        "account_context": {
            "Consumer Discretionary Fund": "Retail, discretionary spend, margins.",
            "Travel & Leisure": "Airlines, hotels, international recovery.",
            "Housing Focus": "Builders, affordability, rates and housing data.",
        },
        "personalized": {
            "Consumer Discretionary Fund": "**Consumer Discretionary Fund**\n• Retail and discretionary supported by sales data; selectivity on margin.\n• Back-to-school and discretionary in focus.",
            "Travel & Leisure": "**Travel & Leisure**\n• Airlines and hotels well positioned; international recovery and pricing power.\n• Demand robust; watch capacity.",
            "Housing Focus": "**Housing Focus**\n• Affordability and rates pressure housing; builders under scrutiny.\n• Existing sales soft; rates key.",
        },
    },
}

def get_briefing_data(vertical: str = None):
    """
    Get briefing data (wires, aggregated briefing, accounts, personalized) for a vertical.
    vertical: One of Technology, Healthcare, Energy, Financials, Consumer. If None, random.
    Returns: dict with wires, aggregated_briefing, accounts, personalized, vertical.
    """
    if vertical is None:
        vertical = random.choice(list(BRIEFING_DATA_CONFIG.keys()))
    if vertical not in BRIEFING_DATA_CONFIG:
        vertical = "Technology"
    config = BRIEFING_DATA_CONFIG[vertical]
    wires = config["wires"]
    aggregated_briefing = config["aggregated_briefing"]
    accounts = config["accounts"]
    account_context_map = config.get("account_context", {})
    n_accounts = random.randint(1, min(3, len(accounts)))
    selected_accounts = random.sample(accounts, n_accounts)
    personalized = {acc: config["personalized"][acc] for acc in selected_accounts}
    # Wires as retrieved documents (id, content, score); content includes time like real wires
    def wire_content(item):
        h, b = item[0], item[1]
        t = item[2] if len(item) > 2 else ""
        return f"[{t}] {h}\n{b}" if t else f"{h}\n{b}"
    wires_as_documents = [
        {"id": f"wire_{vertical}_{i}", "content": wire_content(w), "score": round(random.uniform(0.78, 0.98), 2)}
        for i, w in enumerate(wires)
    ]
    wires_text = "\n---\n".join(f"[Wire {i+1}] {wire_content(w)}" for i, w in enumerate(wires))
    account_context = "\n".join(f"- {acc}: {account_context_map.get(acc, '')}" for acc in selected_accounts)
    return {
        "vertical": vertical,
        "wires": wires,
        "wires_text": wires_text,
        "wires_as_documents": wires_as_documents,
        "aggregated_briefing": aggregated_briefing,
        "accounts": selected_accounts,
        "account_context": account_context,
        "personalized": personalized,
        "personalized_text": "\n\n".join(personalized.values()),
    }


In [6]:
# Bedrock metadata (for span attributes only; no real API calls)
BEDROCK_REGION = "us-east-1"
BEDROCK_MODEL_ID = "claude-3-7-sonnet-latest"

In [7]:
# Briefing Prompt Templates
# Structured prompts for summarize_wires (retrieved wires) and personalize_briefing (briefing + account context)

PROMPT_TEMPLATE_CONFIG = {
    "summarize_wires": {
        "template": (
            "Produce a morning briefing for the **{vertical}** vertical. "
            "Use only the following retrieved wire documents. Do not add facts not present in the wires. "
            "Format: a one-line header (e.g. --- Vertical | Morning Brief ---), then bullet points for key themes/developments, then a single 'Bottom line:' sentence.\n\n"
            "--- Retrieved wires ---\n{wires_text}\n--- End wires ---"
        ),
        "system_message": (
            "You are an analyst briefing assistant. Aggregate wire items into a single briefing. "
            "Output: header line, bullets, then 'Bottom line:' in one sentence. Factual only; suitable for pre-market or client use."
        ),
        "version": "v1.0"
    },
    "personalize_briefing": {
        "template": (
            "Produce a personalized briefing for each account below. "
            "Tailor emphasis to each account's focus. For each account: **Account Name** then 2-3 bullet points.\n\n"
            "Accounts: {account_ids}\n\n"
            "Account context (sector/focus):\n{account_context}\n\n"
            "--- Aggregated briefing to personalize ---\n{briefing}\n--- End briefing ---"
        ),
        "system_message": (
            "You are an analyst briefing assistant. Take the aggregated briefing and personalize for each account. "
            "Same facts; change framing and emphasis. One section per account: **Name** then 2-3 bullets."
        ),
        "version": "v1.0"
    }
}

In [8]:
# Timer utility function
from contextlib import contextmanager
import time

@contextmanager
def timer():
    """Context manager to time execution"""
    start_time = time.time()
    yield
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Execution time: {execution_time:.2f} seconds")

In [9]:
# Briefing Trace Function
# CHAIN -> RETRIEVER -> AGENT -> LLM (summarize) -> guardrailTrace -> ApplyGuardrail (content_filter, pii_filter) -> LLM (personalize) -> guardrailTrace -> ApplyGuardrail (content_filter)

from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import json
import time
import random
from datetime import datetime, timezone

# Specific guards: (name_suffix, input "check for X", exception message when invoked)
GUARD_SPECS = [
    ("content_filter", "check for content policy (harmful / inappropriate content)", "Content policy violation; output blocked"),
    ("pii_filter", "check for PII (personally identifiable information)", "PII detected; content redacted"),
]

def create_briefing_trace(
    tracer,
    vertical: str = None,
    session_id: str = None,
):
    """
    Create a single briefing trace: CHAIN -> RETRIEVER -> AGENT -> LLM (summarize) -> guardrailTrace -> ApplyGuardrail (content_filter, pii_filter) -> LLM (personalize) -> guardrailTrace -> ApplyGuardrail (content_filter).
    Each guard has specific input (check for X issue), output pass/fail; on fail: span status ERROR, exception event, guardrail.status=invoked.
    Returns: dict with span_data including guardrail_span_ids, guardrail_passed_list, guardrail_type_list.
    """
    data = get_briefing_data(vertical)
    vertical = data["vertical"]
    wires_text = data["wires_text"]
    wires_as_documents = data["wires_as_documents"]
    aggregated_briefing = data["aggregated_briefing"]
    accounts = data["accounts"]
    account_context = data["account_context"]
    personalized_text = data["personalized_text"]
    account_ids = ", ".join(accounts)
    summarize_cfg = PROMPT_TEMPLATE_CONFIG.get("summarize_wires", {})
    personalize_cfg = PROMPT_TEMPLATE_CONFIG.get("personalize_briefing", {})
    summarize_system = summarize_cfg.get("system_message", "You are an analyst briefing assistant.")
    summarize_user = summarize_cfg.get("template", "{wires_text}").format(vertical=vertical, wires_text=wires_text)
    personalize_system = personalize_cfg.get("system_message", "Personalize the briefing for each account.")
    personalize_user = personalize_cfg.get("template", "{briefing}").format(
        account_ids=account_ids, account_context=account_context, briefing=aggregated_briefing
    )

    chain_name = "Analyst Briefing Pipeline"
    agent_span_name = "briefing_agent"
    llm_latency_ms_1 = random.uniform(1500, 4000)
    llm_latency_ms_2 = random.uniform(1200, 3500)
    agent_latency_ms = random.uniform(50, 200)
    retrieval_latency_ms = random.uniform(80, 300)
    guardrail_latency_ms = random.uniform(250, 400)
    chain_overhead_ms = random.uniform(20, 100)
    chain_latency_ms = retrieval_latency_ms + agent_latency_ms + llm_latency_ms_1 + 2 * guardrail_latency_ms + llm_latency_ms_2 + chain_overhead_ms

    prompt_tokens_1, completion_tokens_1 = 800, 350
    prompt_tokens_2, completion_tokens_2 = 600, 280
    chain_span_id = retrieval_span_id = agent_span_id = None
    llm_span_ids = []
    guardrail_span_ids = []
    guardrail_passed_list = []
    guardrail_type_list = []

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", f"vertical={vertical}")
        chain_span.set_attribute("input.mime_type", "text/plain")
        if session_id:
            chain_span.set_attribute("session.id", session_id)
        time.sleep(0.005)
        ctx = chain_span.get_span_context()
        chain_span_id = format(ctx.span_id, '016x')

        with tracer.start_as_current_span("wire_fetch") as ret_span:
            ret_span.set_attribute("openinference.span.kind", "RETRIEVER")
            ret_span.set_attribute("input.value", json.dumps({"vertical": vertical, "query": f"wires for {vertical}"}))
            ret_span.set_attribute("input.mime_type", "application/json")
            for idx, doc in enumerate(wires_as_documents):
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.id", doc["id"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
            ret_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in wires_as_documents]))
            ret_span.set_attribute("output.mime_type", "application/json")
            ret_span.set_attribute("latency_ms", round(retrieval_latency_ms, 2))
            ret_span.set_status(Status(StatusCode.OK))
            time.sleep(0.005)
            ctx = ret_span.get_span_context()
            retrieval_span_id = format(ctx.span_id, '016x')

        with tracer.start_as_current_span(agent_span_name) as agent_span:
            agent_span.set_attribute("openinference.span.kind", "AGENT")
            agent_span.set_attribute("agent.name", "briefing_agent")
            agent_input = json.dumps({
                "request": "personalized_analyst_briefing",
                "vertical": vertical,
                "accounts": accounts,
                "instruction": "1) Use retrieved wires 2) Summarize into briefing 3) Personalize for each account.",
            })
            agent_span.set_attribute("input.value", agent_input)
            agent_span.set_attribute("input.mime_type", "application/json")
            if session_id:
                agent_span.set_attribute("session.id", session_id)
            agent_span.set_attribute("latency_ms", round(agent_latency_ms, 2))
            time.sleep(0.005)
            ctx = agent_span.get_span_context()
            agent_span_id = format(ctx.span_id, '016x')

            # LLM 1: summarize_wires (with retrieved wire documents)
            with tracer.start_as_current_span("summarize_wires") as llm1_span:
                if session_id:
                    llm1_span.set_attribute("session.id", session_id)
                llm1_span.set_attribute("openinference.span.kind", "LLM")
                llm1_span.set_attribute("llm.model_name", BEDROCK_MODEL_ID)
                llm1_span.set_attribute("llm.provider", "aws")
                llm1_span.set_attribute("llm.system", "anthropic")
                llm1_span.set_attribute("llm.input_messages.0.message.role", "system")
                llm1_span.set_attribute("llm.input_messages.0.message.content", summarize_system)
                llm1_span.set_attribute("llm.input_messages.1.message.role", "user")
                llm1_span.set_attribute("llm.input_messages.1.message.content", summarize_user)
                llm1_span.set_attribute("llm.output_messages.0.message.role", "assistant")
                llm1_span.set_attribute("llm.output_messages.0.message.content", aggregated_briefing)
                for idx, doc in enumerate(wires_as_documents):
                    llm1_span.set_attribute(f"retrieval.documents.{idx}.document.id", doc["id"])
                    llm1_span.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                    llm1_span.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                llm1_span.set_attribute("llm.token_count.prompt", prompt_tokens_1)
                llm1_span.set_attribute("llm.token_count.completion", completion_tokens_1)
                llm1_span.set_attribute("llm.token_count.total", prompt_tokens_1 + completion_tokens_1)
                llm1_span.set_attribute("input.value", summarize_user[:2000])
                llm1_span.set_attribute("output.value", aggregated_briefing)
                llm1_span.set_attribute("latency_ms", round(llm_latency_ms_1, 2))
                llm1_span.set_status(Status(StatusCode.OK))
                ctx = llm1_span.get_span_context()
                llm_span_ids.append(format(ctx.span_id, '016x'))

                # guardrailTrace -> specific ApplyGuardrail spans after LLM1 (content_filter, pii_filter)
                with tracer.start_as_current_span("guardrailTrace") as gr_trace_span:
                    gr_trace_span.set_attribute("openinference.span.kind", "CHAIN")
                    gr_trace_span.set_attribute("latency_ms", round(2 * guardrail_latency_ms, 2))
                    time.sleep(0.002)
                    for name_suffix, check_desc, exc_msg in GUARD_SPECS:
                        fail = (name_suffix == "pii_filter" and random.random() < 0.15)
                        with tracer.start_as_current_span(f"ApplyGuardrail ({name_suffix})") as g_span:
                            g_span.set_attribute("openinference.span.kind", "GUARDRAIL")
                            g_span.set_attribute("input.value", check_desc)
                            g_span.set_attribute("output.value", "fail" if fail else "pass")
                            g_span.set_attribute("guardrail.status", "invoked" if fail else "not_invoked")
                            g_span.set_attribute("latency_ms", round(guardrail_latency_ms, 2))
                            if fail:
                                g_span.set_status(Status(StatusCode.ERROR, exc_msg))
                                g_span.add_event("exception", attributes={"exception.type": "GuardrailIntervention", "exception.message": exc_msg})
                            else:
                                g_span.set_status(Status(StatusCode.OK))
                            ctx = g_span.get_span_context()
                            guardrail_span_ids.append(format(ctx.span_id, '016x'))
                            guardrail_passed_list.append(not fail)
                            guardrail_type_list.append(name_suffix)

            # LLM 2: personalize_briefing (briefing + account context)
            with tracer.start_as_current_span("personalize_briefing") as llm2_span:
                if session_id:
                    llm2_span.set_attribute("session.id", session_id)
                llm2_span.set_attribute("openinference.span.kind", "LLM")
                llm2_span.set_attribute("llm.model_name", BEDROCK_MODEL_ID)
                llm2_span.set_attribute("llm.provider", "aws")
                llm2_span.set_attribute("llm.system", "anthropic")
                llm2_span.set_attribute("llm.input_messages.0.message.role", "system")
                llm2_span.set_attribute("llm.input_messages.0.message.content", personalize_system)
                llm2_span.set_attribute("llm.input_messages.1.message.role", "user")
                llm2_span.set_attribute("llm.input_messages.1.message.content", personalize_user)
                llm2_span.set_attribute("llm.output_messages.0.message.role", "assistant")
                llm2_span.set_attribute("llm.output_messages.0.message.content", personalized_text)
                llm2_span.set_attribute("llm.token_count.prompt", prompt_tokens_2)
                llm2_span.set_attribute("llm.token_count.completion", completion_tokens_2)
                llm2_span.set_attribute("llm.token_count.total", prompt_tokens_2 + completion_tokens_2)
                llm2_span.set_attribute("input.value", personalize_user[:2000])
                llm2_span.set_attribute("output.value", personalized_text)
                llm2_span.set_attribute("latency_ms", round(llm_latency_ms_2, 2))
                llm2_span.set_status(Status(StatusCode.OK))
                ctx = llm2_span.get_span_context()
                llm_span_ids.append(format(ctx.span_id, '016x'))

                # guardrailTrace -> ApplyGuardrail (content_filter) after LLM2; some fail
                content_spec = GUARD_SPECS[0]
                name_suffix, check_desc, exc_msg = content_spec
                fail_2 = random.random() < 0.05
                with tracer.start_as_current_span("guardrailTrace") as gr_trace_span:
                    gr_trace_span.set_attribute("openinference.span.kind", "CHAIN")
                    gr_trace_span.set_attribute("latency_ms", round(guardrail_latency_ms, 2))
                    time.sleep(0.002)
                    with tracer.start_as_current_span(f"ApplyGuardrail ({name_suffix})") as g_span:
                        g_span.set_attribute("openinference.span.kind", "GUARDRAIL")
                        g_span.set_attribute("input.value", check_desc)
                        g_span.set_attribute("output.value", "fail" if fail_2 else "pass")
                        g_span.set_attribute("guardrail.status", "invoked" if fail_2 else "not_invoked")
                        g_span.set_attribute("latency_ms", round(guardrail_latency_ms, 2))
                        if fail_2:
                            g_span.set_status(Status(StatusCode.ERROR, exc_msg))
                            g_span.add_event("exception", attributes={"exception.type": "GuardrailIntervention", "exception.message": exc_msg})
                        else:
                            g_span.set_status(Status(StatusCode.OK))
                        ctx = g_span.get_span_context()
                        guardrail_span_ids.append(format(ctx.span_id, '016x'))
                        guardrail_passed_list.append(not fail_2)
                        guardrail_type_list.append(name_suffix)

            # Agent output = final personalized briefing (tied to last LLM call)
            agent_span.set_attribute("output.value", personalized_text)
            agent_span.set_attribute("output.mime_type", "text/plain")

        chain_span.set_attribute("output.value", personalized_text[:500])
        chain_span.set_attribute("output.mime_type", "text/plain")
        chain_span.set_attribute("latency_ms", round(chain_latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

    return {
        "span_data": {
            "chain_span_id": chain_span_id,
            "retrieval_span_id": retrieval_span_id,
            "agent_span_id": agent_span_id,
            "llm_span_ids": llm_span_ids,
            "guardrail_span_ids": guardrail_span_ids,
            "guardrail_passed_list": guardrail_passed_list,
            "guardrail_type_list": guardrail_type_list,
            "vertical": vertical,
            "query": f"vertical={vertical}",
            "response": personalized_text,
            "aggregated_briefing": aggregated_briefing,
            "wires_text": wires_text,
            "accounts": accounts,
            "session_id": session_id,
        }
    }


In [10]:
# Session Evaluation Helper Function
# Processes CHAIN spans grouped by session to generate session evaluations

def generate_session_evaluations_from_chain_spans(span_df: pd.DataFrame) -> list:
    """
    Generate session evaluations from CHAIN spans.
    Groups by session_id, gets the oldest CHAIN span per session, and generates evaluations.
    
    Args:
        span_df: DataFrame containing span data with CHAIN spans
        
    Returns:
        list: List of evaluation dictionaries with session_eval.* columns
    """
    session_evaluations = []
    
    # Filter for CHAIN spans only
    chain_spans = span_df[span_df['openinference.span.kind'] == 'CHAIN'].copy()
    
    if len(chain_spans) == 0:
        return session_evaluations
    
    # Check if session_id column exists
    if 'session_id' not in chain_spans.columns:
        print("⚠️  No session_id column found in CHAIN spans. Skipping session evaluations.")
        return session_evaluations
    
    # Add timestamp if not present (use index as proxy for order)
    if 'timestamp' not in chain_spans.columns:
        chain_spans['timestamp'] = chain_spans.index
    
    # Remove rows with null session_id
    chain_spans = chain_spans[chain_spans['session_id'].notna()]
    
    if len(chain_spans) == 0:
        return session_evaluations
    
    # Group by session_id and get the oldest (first) CHAIN span per session
    # Sort by timestamp and take first per group
    chain_spans_sorted = chain_spans.sort_values('timestamp')
    oldest_chain_per_session = chain_spans_sorted.groupby('session_id').first().reset_index()
    
    print(f"  Processing {len(oldest_chain_per_session):,} unique sessions for session evaluations")
    
    # Generate session evaluations for oldest CHAIN spans
    for _, row in oldest_chain_per_session.iterrows():
        span_id = row.get('context.span_id', '')
        is_poisoned = row.get('is_poisoned', False)
        session_id = row.get('session_id', 'unknown')
        
        span_data = {
            'vertical': row.get('vertical', row.get('scenario', '')),
            'scenario': row.get('scenario', row.get('vertical', '')),
            'query': row.get('query', ''),
            'response': row.get('response', ''),
            'session_id': session_id
        }
        
        # Generate session evaluation (briefing pipeline)
        will_pass = random.random() < 0.82  # 82% pass rate
        vertical = span_data.get('vertical', span_data.get('scenario', 'Technology'))
        
        if will_pass:
            label = "pass"
            score = 0
            explanations = [
                f"Briefing pipeline passed: The complete session for {vertical} (session_id: {session_id}) produced a coherent briefing. Wire fetch, summarization, guardrails, and personalization worked together cohesively.",
                f"Session validation successful: The end-to-end briefing pipeline for {vertical} was executed correctly. The chain orchestrated retrieval, LLM, and guardrail components effectively.",
                f"Briefing quality verified: The session (session_id: {session_id}) for {vertical} demonstrated proper coordination. Aggregated briefing and personalized output met quality standards."
            ]
        else:
            label = "fail"
            score = 1
            explanations = [
                f"Briefing pipeline failed: The session for {vertical} (session_id: {session_id}) encountered issues in coordination or execution. Components did not work together effectively.",
                f"Session validation failed: The briefing pipeline for {vertical} had coordination problems. The chain failed to properly orchestrate retrieval, LLM, or guardrail components.",
                f"Briefing quality check failed: The session (session_id: {session_id}) for {vertical} showed misalignment between components (wire fetch, summarization, personalization, or guardrails)."
            ]
        
        eval_data = {
            "label": label,
            "score": score,
            "explanation": random.choice(explanations)
        }
        
        session_evaluations.append({
            'context.span_id': span_id,
            'session_eval.label': eval_data['label'],
            'session_eval.score': eval_data['score'],
            'session_eval.explanation': eval_data['explanation']
        })
    
    return session_evaluations

In [11]:
# Batch collection includes CHAIN, AGENT, LLM (with llm_step); guardrail spans collected but no evals generated.
# Session evaluations use CHAIN spans grouped by session_id; briefing evals use vertical.
print("📝 Batch cell collects CHAIN, AGENT, LLM for evaluation generation (no evals on guardrails).")

📝 Batch cell collects CHAIN, AGENT, LLM for evaluation generation (no evals on guardrails).


In [12]:
# Evaluation Generation Functions
# Briefing pipeline: AGENT, LLM (summarize/personalize) evals; no guardrail evals

import pandas as pd
import random

def generate_agent_trajectory_accuracy_eval(span_data: dict) -> dict:
    """Briefing orchestration: did the agent sequence summarize -> personalize correctly?"""
    will_pass = random.random() < 0.75
    vertical = span_data.get('vertical', span_data.get('scenario', 'Technology'))
    if will_pass:
        label, score = "pass", 0
        explanations = [
            f"Briefing orchestration passed: The agent correctly sequenced wire fetch, summarization, guardrails, and personalization for {vertical}.",
            f"Agent trajectory validation successful: The agent followed the correct sequence (summarize -> personalize) for the {vertical} briefing pipeline.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            f"Briefing orchestration failed: The agent deviated from expected sequence for {vertical}.",
            f"Agent trajectory validation failed: The agent made suboptimal decisions during the {vertical} briefing pipeline.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_hallucination_eval(span_data: dict) -> dict:
    """Hallucination evaluation for LLM spans."""
    will_pass = random.random() < 0.85
    vertical = span_data.get('vertical', span_data.get('scenario', 'Technology'))
    if will_pass:
        label, score = "pass", 0
        explanations = [
            f"Hallucination check passed: The LLM response for {vertical} is factually consistent and does not contain fabricated information.",
            f"No hallucination detected: The response accurately addresses the {vertical} briefing without introducing unsupported claims.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            f"Hallucination detected: The LLM response for {vertical} contains fabricated or unsupported information.",
            f"Factual inconsistency found: The response to the {vertical} briefing includes hallucinated details.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_vertical_relevance_eval(span_data: dict) -> dict:
    """Does the aggregated briefing stay on-topic for the requested vertical? (LLM summarize step)"""
    will_pass = random.random() < 0.88
    vertical = span_data.get('vertical', span_data.get('scenario', 'Technology'))
    if will_pass:
        label, score = "pass", 0
        explanations = [f"Vertical relevance passed: The aggregated briefing for {vertical} stays on-topic and aligns with the vertical.",]
    else:
        label, score = "fail", 1
        explanations = [f"Vertical relevance failed: The aggregated briefing for {vertical} drifts off-topic or lacks vertical alignment.",]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_personalization_quality_eval(span_data: dict) -> dict:
    """Is the personalized output appropriate for the listed accounts? (LLM personalize step)"""
    will_pass = random.random() < 0.85
    vertical = span_data.get('vertical', span_data.get('scenario', 'Technology'))
    if will_pass:
        label, score = "pass", 0
        explanations = [f"Personalization quality passed: The personalized briefing for {vertical} is appropriate for the listed accounts.",]
    else:
        label, score = "fail", 1
        explanations = [f"Personalization quality failed: The personalized briefing for {vertical} does not adequately tailor to the accounts.",]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def create_evaluations_from_span_data(span_df: pd.DataFrame) -> pd.DataFrame:
    """Create evaluation DataFrame from collected span data. Handles CHAIN (skip), AGENT, LLM (step). No evals on guardrails."""
    evaluations = []
    for _, row in span_df.iterrows():
        span_kind = row.get('openinference.span.kind', '')
        span_id = row.get('context.span_id', '')
        span_data = {
            'vertical': row.get('vertical', row.get('scenario', '')),
            'scenario': row.get('scenario', row.get('vertical', '')),
            'query': row.get('query', ''),
            'response': row.get('response', ''),
            'guardrail_passed': row.get('guardrail_passed', True),
            'guardrail_type': row.get('guardrail_type', 'content_filter'),
        }
        if span_kind == 'AGENT':
            eval_data = generate_agent_trajectory_accuracy_eval(span_data)
            evaluations.append({
                'context.span_id': span_id,
                'trace_eval.AgentTrajectoryAccuracy.label': eval_data['label'],
                'trace_eval.AgentTrajectoryAccuracy.score': eval_data['score'],
                'trace_eval.AgentTrajectoryAccuracy.explanation': eval_data['explanation']
            })
        elif span_kind == 'LLM':
            llm_step = row.get('llm_step', '')
            eval_data = generate_hallucination_eval(span_data)
            evaluations.append({
                'context.span_id': span_id,
                'eval.Hallucination.label': eval_data['label'],
                'eval.Hallucination.score': eval_data['score'],
                'eval.Hallucination.explanation': eval_data['explanation']
            })
            if llm_step == 'summarize_wires':
                vr = generate_vertical_relevance_eval(span_data)
                evaluations.append({
                    'context.span_id': span_id,
                    'eval.VerticalRelevance.label': vr['label'],
                    'eval.VerticalRelevance.score': vr['score'],
                    'eval.VerticalRelevance.explanation': vr['explanation']
                })
            elif llm_step == 'personalize_briefing':
                pq = generate_personalization_quality_eval(span_data)
                evaluations.append({
                    'context.span_id': span_id,
                    'eval.PersonalizationQuality.label': pq['label'],
                    'eval.PersonalizationQuality.score': pq['score'],
                    'eval.PersonalizationQuality.explanation': pq['explanation']
                })
    return pd.DataFrame(evaluations)


# Single Briefing Trace Test

In [13]:
# Sandbox: Test single briefing trace creation
# CHAIN -> RETRIEVER -> AGENT -> LLM (summarize) -> GUARDRAIL x2 -> LLM (personalize) -> GUARDRAIL x1

from opentelemetry import trace

tracer = trace.get_tracer(__name__)

with timer():
    result = create_briefing_trace(tracer=tracer, vertical="Technology", session_id="test_session_001")
    sd = result["span_data"]
    print(f"Created briefing trace for vertical: {sd['vertical']}")
    print(f"  chain_span_id: {sd['chain_span_id']}, agent_span_id: {sd['agent_span_id']}")
    print(f"  llm_span_ids: {sd['llm_span_ids']}, guardrail_span_ids: {sd['guardrail_span_ids']}")
    print("Created test trace – check in Arize for data.")

    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=30000)
    if flush_success:
        print("✅ Single trace exported successfully")
    else:
        print("⚠️  Flush timeout - span may not have been exported")


Created briefing trace for vertical: Technology
  chain_span_id: 1adc449c06ea6aa6, agent_span_id: a51207a9b5e1c712
  llm_span_ids: ['12e6b129c40454f2', '26715b665f61adc5'], guardrail_span_ids: ['76cf955c46c6adbc', '55df00db258ae97a', 'b8e3f69894fd35f7']
Created test trace – check in Arize for data.
✅ Single trace exported successfully
Execution time: 0.50 seconds


# Full batch briefing trace creation

In [14]:
# Batch generation: create_briefing_trace for each run; collect CHAIN, AGENT, LLM (with step) for evals

from opentelemetry import trace

tracer = trace.get_tracer(__name__)
VERTICALS = ["Technology", "Healthcare", "Energy", "Financials", "Consumer"]

NUM_SPANS = 500
print(f"Generating {NUM_SPANS:,} briefing traces (CHAIN -> RETRIEVER -> AGENT -> LLM -> guardrailTrace->Guardrails -> LLM -> guardrailTrace->Guardrails)...")
print("Note: BatchSpanProcessor batches automatically.\n")

with timer():
    span_data_list = []
    SESSION_SIZE_MIN = 2
    SESSION_SIZE_MAX = 5
    current_session_id = None
    traces_in_current_session = 0
    session_size = 0
    BATCH_SIZE = 50

    for batch_start in range(0, NUM_SPANS, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, NUM_SPANS)
        for i in range(batch_start, batch_end):
            if current_session_id is None or traces_in_current_session >= session_size:
                current_session_id = f"session_{random.randint(100000, 999999)}"
                session_size = random.randint(SESSION_SIZE_MIN, SESSION_SIZE_MAX)
                traces_in_current_session = 0
            vertical = random.choice(VERTICALS)
            result = create_briefing_trace(tracer=tracer, vertical=vertical, session_id=current_session_id)
            traces_in_current_session += 1

            if "span_data" in result:
                sd = result["span_data"]
                # CHAIN
                span_data_list.append({
                    "context.span_id": sd["chain_span_id"],
                    "openinference.span.kind": "CHAIN",
                    "vertical": sd["vertical"],
                    "scenario": sd["vertical"],
                    "query": sd["query"],
                    "response": sd["response"],
                    "session_id": sd.get("session_id"),
                })
                # AGENT
                span_data_list.append({
                    "context.span_id": sd["agent_span_id"],
                    "openinference.span.kind": "AGENT",
                    "vertical": sd["vertical"],
                    "query": sd["query"],
                    "response": sd["response"],
                })
                # LLM summarize_wires
                span_data_list.append({
                    "context.span_id": sd["llm_span_ids"][0],
                    "openinference.span.kind": "LLM",
                    "llm_step": "summarize_wires",
                    "vertical": sd["vertical"],
                    "query": sd.get("wires_text", "")[:300],
                    "response": sd.get("aggregated_briefing", ""),
                })
                # LLM personalize_briefing
                span_data_list.append({
                    "context.span_id": sd["llm_span_ids"][1],
                    "openinference.span.kind": "LLM",
                    "llm_step": "personalize_briefing",
                    "vertical": sd["vertical"],
                    "query": sd.get("aggregated_briefing", "")[:300],
                    "response": sd["response"],
                })
                # GUARDRAIL spans (3 per trace: content_filter + pii_filter after summarize, content_filter after personalize)
                passed_list = sd.get("guardrail_passed_list", [True] * len(sd["guardrail_span_ids"]))
                type_list = sd.get("guardrail_type_list", ["content_filter"] * len(sd["guardrail_span_ids"]))
                for gid, passed, gtype in zip(sd["guardrail_span_ids"], passed_list, type_list):
                    span_data_list.append({
                        "context.span_id": gid,
                        "openinference.span.kind": "GUARDRAIL",
                        "guardrail_passed": passed,
                        "guardrail_type": gtype,
                    })

        if (batch_end % 500 == 0) or (batch_end == NUM_SPANS):
            print(f"  Created {batch_end:,} / {NUM_SPANS:,} traces...")

    print(f"\nCompleted creating {NUM_SPANS:,} traces. Flushing remaining spans...")
    base_timeout = 30000
    flush_timeout = min(base_timeout + (NUM_SPANS // 10000) * 10000, 300000)
    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
    if flush_success:
        print(f"✅ Flush successful. Collected {len(span_data_list):,} span records for evaluation generation.")
    else:
        print("⚠️  Flush timeout – some spans may not have been exported.")

collected_span_data = span_data_list


Generating 500 briefing traces (CHAIN -> RETRIEVER -> AGENT -> LLM -> guardrailTrace->Guardrails -> LLM -> guardrailTrace->Guardrails)...
Note: BatchSpanProcessor batches automatically.

  Created 500 / 500 traces...

Completed creating 500 traces. Flushing remaining spans...
✅ Flush successful. Collected 3,500 span records for evaluation generation.
Execution time: 13.72 seconds


In [15]:
# Evaluation Generation and Logging
# This cell generates evaluations from collected span data and logs them to Arize
# Run this cell AFTER the span generation cell has completed

print(f"{'='*60}")
print("Generating evaluations from collected span data...")
print(f"{'='*60}\n")

if 'collected_span_data' not in globals() or not collected_span_data:
    print("⚠️  No span data found. Please run the span generation cell first.")
else:
    # Convert span data to DataFrame
    span_df = pd.DataFrame(collected_span_data)
    print(f"Collected {len(span_df):,} spans")
    print(f"  - CHAIN spans: {len(span_df[span_df['openinference.span.kind'] == 'CHAIN']):,}")
    print(f"  - AGENT spans: {len(span_df[span_df['openinference.span.kind'] == 'AGENT']):,}")
    print(f"  - LLM spans: {len(span_df[span_df['openinference.span.kind'] == 'LLM']):,}")
    print(f"  - GUARDRAIL spans: {len(span_df[span_df['openinference.span.kind'] == 'GUARDRAIL']):,}")
    
    # Generate evaluations for AGENT, LLM spans (no guardrail evals)
    print(f"\nGenerating evaluations for AGENT, LLM spans...")
    eval_df = create_evaluations_from_span_data(span_df)
    
    # Generate session evaluations from CHAIN spans (grouped by session)
    print(f"\nGenerating session evaluations from CHAIN spans...")
    session_evaluations = generate_session_evaluations_from_chain_spans(span_df)
    
    # Merge session evaluations with existing evaluations
    if session_evaluations:
        session_eval_df = pd.DataFrame(session_evaluations)
        eval_df = pd.concat([eval_df, session_eval_df], ignore_index=True)
        print(f"  Added {len(session_evaluations):,} session evaluations")
    
    if len(eval_df) > 0:
        print(f"✅ Generated {len(eval_df):,} evaluations\n")
        
        # Show evaluation breakdown
        agent_evals = len(eval_df[eval_df['trace_eval.AgentTrajectoryAccuracy.label'].notna()]) if 'trace_eval.AgentTrajectoryAccuracy.label' in eval_df.columns else 0
        llm_evals = len(eval_df[eval_df['eval.Hallucination.label'].notna()]) if 'eval.Hallucination.label' in eval_df.columns else 0
        session_evals = len(eval_df[eval_df['session_eval.label'].notna()]) if 'session_eval.label' in eval_df.columns else 0
        vr_evals = len(eval_df[eval_df['eval.VerticalRelevance.label'].notna()]) if 'eval.VerticalRelevance.label' in eval_df.columns else 0
        pq_evals = len(eval_df[eval_df['eval.PersonalizationQuality.label'].notna()]) if 'eval.PersonalizationQuality.label' in eval_df.columns else 0
        print(f"  - AgentTrajectoryAccuracy: {agent_evals:,}")
        print(f"  - Hallucination: {llm_evals:,}")
        print(f"  - VerticalRelevance: {vr_evals:,}")
        print(f"  - PersonalizationQuality: {pq_evals:,}")
        print(f"  - Session evaluations: {session_evals:,}")
        
        # Show pass/fail distribution
        if 'trace_eval.AgentTrajectoryAccuracy.label' in eval_df.columns:
            agent_pass = (eval_df['trace_eval.AgentTrajectoryAccuracy.label'] == 'pass').sum()
            agent_fail = (eval_df['trace_eval.AgentTrajectoryAccuracy.label'] == 'fail').sum()
            print(f"    AgentTrajectoryAccuracy: {agent_pass:,} pass, {agent_fail:,} fail")
        
        if 'eval.Hallucination.label' in eval_df.columns:
            llm_pass = (eval_df['eval.Hallucination.label'] == 'pass').sum()
            llm_fail = (eval_df['eval.Hallucination.label'] == 'fail').sum()
            print(f"    Hallucination: {llm_pass:,} pass, {llm_fail:,} fail")
        
        if 'session_eval.label' in eval_df.columns:
            session_pass = (eval_df['session_eval.label'] == 'pass').sum()
            session_fail = (eval_df['session_eval.label'] == 'fail').sum()
            print(f"    Session evaluation: {session_pass:,} pass, {session_fail:,} fail")
        if 'eval.VerticalRelevance.label' in eval_df.columns:
            vr_pass = (eval_df['eval.VerticalRelevance.label'] == 'pass').sum()
            vr_fail = (eval_df['eval.VerticalRelevance.label'] == 'fail').sum()
            print(f"    VerticalRelevance: {vr_pass:,} pass, {vr_fail:,} fail")
        if 'eval.PersonalizationQuality.label' in eval_df.columns:
            pq_pass = (eval_df['eval.PersonalizationQuality.label'] == 'pass').sum()
            pq_fail = (eval_df['eval.PersonalizationQuality.label'] == 'fail').sum()
            print(f"    PersonalizationQuality: {pq_pass:,} pass, {pq_fail:,} fail")
        
        # Log evaluations to Arize
        print(f"\n{'='*60}")
        print("Logging evaluations to Arize...")
        print(f"{'='*60}\n")
        
        try:
            from arize.pandas.logger import Client
            client_kwargs = {
                "space_id": space_id, 
                "api_key": api_key
            }
            # Only add URI if endpoint is defined
            if 'endpoint' in globals() and endpoint:
                client_kwargs["URI"] = endpoint
            client = Client(**client_kwargs)
            client.log_evaluations_sync(eval_df, project_name)
            print(f"✅ Successfully logged {len(eval_df):,} evaluations to Arize")
            print(f"   Check the Arize dashboard to verify receipt.")
        except Exception as e:
            print(f"⚠️  Error logging evaluations: {e}")
            print(f"   Evaluation DataFrame is available as 'eval_df' variable.")
            print(f"   You can log it manually later or fix the error and retry.")
    else:
        print("⚠️  No evaluations generated. Check span data collection.")

Generating evaluations from collected span data...

Collected 3,500 spans
  - CHAIN spans: 500
  - AGENT spans: 500
  - LLM spans: 1,000
  - GUARDRAIL spans: 1,500

Generating evaluations for AGENT, LLM spans...

Generating session evaluations from CHAIN spans...
  Processing 144 unique sessions for session evaluations
  Added 144 session evaluations
✅ Generated 2,644 evaluations

  - AgentTrajectoryAccuracy: 500
  - Hallucination: 1,000
  - VerticalRelevance: 500
  - PersonalizationQuality: 500
  - Session evaluations: 144
    AgentTrajectoryAccuracy: 372 pass, 128 fail
    Hallucination: 865 pass, 135 fail
    Session evaluation: 117 pass, 27 fail
    VerticalRelevance: 421 pass, 79 fail
    PersonalizationQuality: 429 pass, 71 fail

Logging evaluations to Arize...

  arize.utils.logging | INFO | The following columns do not follow the evaluation column naming convention and will be ignored: session_eval.label, session_eval.score and session_eval.explanation. Evaluation columns must 